# Pipeline extraction images et annotation automatique

### Pipeline:

- Chargement du PDF
- Création d'images à partir de chaque page du PDF
- Inférence avec Yolo sur chaque PDF pour détecter les cadres des images
- Recadrage des images et enregistrement
- Annotation des images avec l'API OpenRouter et enregistrement au format JSON avec le nom de l'image et le nom du PDF d'origine (important pour examiner de près certains livres, comme « Sans famille », et comparer les déductions visuelles avec celles issues du texte !)

## 1. Création d'images à partir de chaque page du PDF

In [ ]:
# funzione che prende in input un pdf e trasforma ogni pagina in un'immagine
import os
from pdf2image import convert_from_path

def pdf_to_images(pdf_path, output_folder, dpi=150):
    # controlla se la cartella di output esiste, altrimenti la crea
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # converte il pdf in immagini
    images = convert_from_path(pdf_path)

    # salva ogni immagine nella cartella di output
    for i, image in enumerate(images):
        image.save(os.path.join(output_folder, f'page_{i + 1}.jpg'), 'JPEG')

# esempio di utilizzo
if __name__ == "__main__":
    pdf_path = 'pdf'  # percorso del file PDF da convertire
    output_folder = 'pdf_da_annotare'  # cartella di output per le immagini
    pdf_to_images(pdf_path, output_folder)
    print(f'PDF convertito in immagini e salvato nella cartella "{output_folder}".')



python(6698) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(6699) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(6700) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


PDF convertito in immagini e salvato nella cartella "/Users/deniseatzori/Library/Mobile Documents/com~apple~CloudDocs/ENC-PSL/Cours/Traitement image thematique/pdf_da_annotare".


##   2. Inférence avec Yolo sur chaque fichier PDF pour détecter les zones de détection des images et enregistrement

Chargement du modèle YOLO entraîné

In [11]:
import os
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from pdf2image import convert_from_path
from ultralytics import YOLO
import gc

In [12]:
def extract_illustrations(input_folder, output_base_folder, model_path='illustration_detector4.pt'):
    # Caricamento modello
    model = YOLO(model_path)
    
    input_path = Path(input_folder)
    output_path = Path(output_base_folder)
    output_path.mkdir(parents=True, exist_ok=True)

    # Lista dei PDF da processare
    pdf_files = list(input_path.glob("*.pdf"))
    
    # Barra di progresso principale (per i libri)
    for pdf_file in tqdm(pdf_files, desc="Libri totali", unit="book"):
        book_output_dir = output_path / pdf_file.stem

        # ✅ NUOVO CONTROLLO: se la cartella esiste già → salta
        if book_output_dir.exists():
            print(f"[SKIP] {pdf_file.name} già processato.")
            continue

        # Crea la cartella solo se non esiste
        book_output_dir.mkdir(parents=True, exist_ok=True)
        
        try:
            pages = convert_from_path(pdf_file, dpi=150)
            
            for pg_num, page_img in enumerate(
                tqdm(pages, desc=f"Pagine {pdf_file.stem[:15]}...", leave=False), start=1
            ):
                results = model(page_img, verbose=False, conf=0.25) 
                
                for i, result in enumerate(results):
                    boxes = result.boxes.xyxy.cpu().numpy()
                    
                    for j, box in enumerate(boxes):
                        illustration = page_img.crop((box[0], box[1], box[2], box[3]))
                        filename = f"pg_{pg_num}_img_{j+1}.jpg"
                        illustration.save(book_output_dir / filename, "JPEG", quality=85)
                
                page_img.close()

            del pages
            gc.collect()

        except Exception as e:
            print(f"\n[ERRORE] Salto {pdf_file.name}: {e}")

In [17]:
extract_illustrations('prova', 'illustrations')

Libri totali: 100%|██████████| 1/1 [00:00<00:00, 8774.69book/s]

[SKIP] Grand_mère_avait_des_défauts.pdf già processato.


## 3. Annotation avec un LLM

### 3.1 Envoi de requêtes à open router (LLM)

In [1]:
import base64

# funzione che prende in input il percorso di un'immagine e restituisce la stringa base64
def encode_image_base64(image_path):
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

In [2]:
# funzione che prende in input una lista di immagini (con path, nome file e sottocartella) e le invia a OpenRouter per l'analisi, restituendo i risultati in una lista

import requests
import time
import json

def invia_richiesta_openrouter(img_path, system_prompt, api_key, model="gpt-5-mini-2025-08-07"):
    url = "https://openrouter.ai/api/v1/chat/completions"

    # prepara le intestazioni per l'autenticazione e il tipo di contenuto
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    print(img_path)
    base64_img = encode_image_base64(img_path)

    # costruisce il payload per la richiesta
    payload = {
        "model": model,
        "messages": [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": "Analyse this image and answer in JSON format."
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{base64_img}"
                        }
                    }
                ]
            }
        ],
        "max_tokens" : 2000,
        "max_output_tokens": 300,
        "temperature" : 0,
        "response_format" : {"type" : "json_object"},
        "reasoning": { "effort": "low"},
        #"top_p": 0,
    }

    s = random.uniform(0.5, 1.5)
    print(f"sto per inviare la richiesta per {img_path}, ma aspetto {s} secondi")
    # pausa casuale tra 2 e 3 secondi per evitare rate limit
    time.sleep(2 + s)
    
    # invia la richiesta a OpenRouter
    response = requests.post(url, headers=headers, json=payload)
    print(response.json())
    # controlla se la richiesta è andata a buon fine
    if response.status_code != 200:
        print(f"Errore su {img_path}: {response.text}")
        raise Exception
        return None

    # estrae la risposta in formato JSON
    risposta = response.json()

# estrai il contenuto della risposta (assumendo che sia in choices[0].message.content)
    try:
        contenuto = risposta["choices"][0]["message"]["content"]
    except Exception:
        contenuto = None

    return json.loads(contenuto)

In [3]:

# funzione che prende in input i risultati dell'analisi (con subfolder, file_name e llm_output) e li salva in un csv strutturato, appiattendo il JSON restituito da LLM e creando una colonna per ogni chiave (composte) del JSON

def salva_json_borbott(risultato, output_path):

    if risultato is None or risultato["llm_output"] is None:
        return

    with open(output_path, "a", encoding="utf-8") as f:
        json.dump(risultato, f)
        f.write(",")


In [ ]:
ROOT_DIR = "illustrations_pulite"
FATTE_DIR = "fatte"

# N = 10
API_KEY = "xxxx"

SYSTEM_PROMPT = """You are given a children's book illustration. Analyze the image and answer the following 6 questions based on the visual content only. Respond using the exact JSON format provided.
Questions:
1. Is there an illustration on this page? Answer 'Yes' or 'No'. If 'No', skip the remaining questions and respond with 'None' for all other fields.
2. Who are the characters on the page? Choose all that apply from (if multiple characters of the same type are present, list them all):
['Man', 'Woman', 'Girl', 'Boy', 'Animal', 'Animate Object', 'Mythical / Imaginary Being',
'Other', 'None']
3. Where is the scene set? Choose one: 'Indoors', 'Outdoors', or 'Both / Ambiguous'
4. What are the faces position? For every character, choose one for each character (list them in the same order of the characters in question): 'front', 'three_quarter', 'profile', 'three_quarter_look_up', 'three_quarter_look_down', 'None'
5. What social class do the characters belong to? choose one for each character (list them in the same order of the characters in question): 'Upper Class', 'Middle Class', 'Working Class', 'Poor', 'None'
6. Are there any toys in the scene? Select all that apply: ['doll', 'toy', 'train', 'toy soldier', 'rocking horse', 'hoop', 'puppet', 'other', 'None']
7. How emotionally intense is the scene? Rate from 1 (very calm) to 5 (very intense).
Respond in this exact JSON format (no explanations or extra commentary):
{
"illustration_present": "",
"characters": [],
"scene_setting": "",
"faces_position": [],
"social_class": [],
"toy": [],
"emotional_intensity":
}
Answer immediately. Do not think step-by-step.
Do not analyze. Just output the JSON.
"""

In [5]:
import shutil
import os
import random

def processa_e_salva(root_dir, system_prompt, api_key, out_filename):
    estensioni = (".jpg", ".jpeg", ".png")
    
    for root, _, files in os.walk(root_dir):
        for file in files:
            if file.lower().endswith(estensioni):
                
                immagine = os.path.join(root, file)
                subfolder = os.path.basename(root)
                sottocartella = os.path.relpath(root, root_dir)
                
                print(f"libro: {subfolder}, immagine: {file} pronta all'analisi ")
                
                res = invia_richiesta_openrouter(immagine, system_prompt, api_key)
                res_completa = {
                    "subfolder": subfolder,
                    "file_name": file,
                    "llm_output": res
                }
                
                print(f"libro: {subfolder}, immagine: {file} analizzata ")

                salva_json_borbott(res_completa, out_filename)
                
                print(f"libro: {subfolder}, immagine: {file} scritta ")
                
                sottocartella_fatte = os.path.join(FATTE_DIR, sottocartella)
                os.makedirs(sottocartella_fatte, exist_ok=True)
                immagine_fatte = os.path.join(sottocartella_fatte, file)
                shutil.move(immagine, immagine_fatte)
                
                print(f"libro: {subfolder}, immagine: {file} spostata")

In [7]:
processa_e_salva(ROOT_DIR, SYSTEM_PROMPT, API_KEY, "risultati_analisi.json")

libro: 1890_Voyages_abracadabrants_du_gros_Philéas, immagine: pg_41_img_1.jpg pronta all'analisi 
/Users/deniseatzori/Library/Mobile Documents/com~apple~CloudDocs/ENC-PSL/Cours/Traitement image thematique/illustrations_pulite/1890_Voyages_abracadabrants_du_gros_Philéas/pg_41_img_1.jpg
sto per inviare la richiesta per /Users/deniseatzori/Library/Mobile Documents/com~apple~CloudDocs/ENC-PSL/Cours/Traitement image thematique/illustrations_pulite/1890_Voyages_abracadabrants_du_gros_Philéas/pg_41_img_1.jpg, ma aspetto 1.2294913309151736 secondi
{'id': 'gen-1775985046-KwczBjyQEsIJWF7ZOv3v', 'object': 'chat.completion', 'created': 1775985046, 'model': 'openai/gpt-5-mini-2025-08-07', 'provider': 'OpenAI', 'system_fingerprint': None, 'service_tier': 'default', 'choices': [{'index': 0, 'logprobs': None, 'finish_reason': 'stop', 'native_finish_reason': 'completed', 'message': {'role': 'assistant', 'content': '{\n"illustration_present": "Yes",\n"characters": ["Boy", "Boy", "Man", "Animal"],\n"s